 Train + test Z-Image LoRA
Train a Z-Image LoRA from a Drive dataset, then compare epochs.

1. **Runtime > Change runtime type > T4 GPU**
2. **Secrets** (key icon, left), allow notebook access: `HF_TOKEN` (gated or private repos),
   `CIVITAI_TOKEN` (Civitai downloads), `TS_AUTHKEY` (Tailscale, tick Reusable + Ephemeral; only the app notebooks).
3. Run **Setup** (re-runs skip finished steps), then the cells below in order.

In [ ]:
#@title Setup: repo, venv, secrets, Drive
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/kopeng-mi/SDXL_Research.git"  #@param {type:"string"}
BRANCH = "step5-notebooks"  #@param {type:"string"}
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/SDXLR"  #@param {type:"string"}
GROUPS = 'caption,train-z'  # venv install groups for this notebook (None = no shared venv)
REPO = Path("/content/SDXL_Research")

if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "--recurse-submodules", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    # follow BRANCH even if an earlier run cloned another one
    subprocess.run(["git", "fetch", "-q", "origin", BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"], cwd=REPO, check=True)
    subprocess.run(["git", "submodule", "update", "-q", "--init", "--recursive"], cwd=REPO, check=True)

# The kernel only orchestrates (stdlib); GPU work runs in venv subprocesses,
# so this kernel never holds VRAM.
sys.path.insert(0, str(REPO / "src"))
from sdxlr import nb
if GROUPS is not None:
    r = subprocess.run([sys.executable, "-m", "sdxlr.bootstrap", "--groups", GROUPS], stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, text=True, env={**os.environ, "PYTHONPATH": str(REPO / "src")})
    print(r.stdout[-3000:])
    assert r.returncode == 0, "setup failed (see above)"
found = nb.load_secrets()
print("secrets:", ", ".join(f"{k} {'found' if v else 'missing'}" for k, v in found.items()))
ROOT = nb.drive_mount(DRIVE_ROOT) if USE_DRIVE else None
print("drive:", ROOT or "off")

# 1. Dataset
Builds the dataset on the VM from any mix of: the Drive dataset `SDXLR/datasets/<DS_NAME>`, a **.zip you upload**
(tick UPLOAD_ZIP; images with their .txt captions, folders inside are fine), and SRC links.
Duplicates are skipped and phone photos are turned upright. The result is copied back to Drive.
Photos without a caption can be captioned here (Qwen3-VL 4B, identity style, 80 to 150 words).

In [ ]:
#@title Dataset
DS_NAME = "my_dataset"  #@param {type:"string"}
UPLOAD_ZIP = False  #@param {type:"boolean"}
SRC = ""  #@param {type:"string"}
#@markdown SRC: comma list of .zip links, HF files `org/repo/file.zip`, or Drive paths (optional).
CAPTION_MISSING = True  #@param {type:"boolean"}
TRIGGER = "ohwx"  #@param {type:"string"}
CLASS_WORD = "woman"  #@param {type:"string"}
from pathlib import Path
from sdxlr import drive, plan, paths
IMGS = (".png", ".jpg", ".jpeg", ".webp")
LOCAL = paths.DATASETS / DS_NAME
LOCAL.mkdir(parents=True, exist_ok=True)
DRIVE_DS = ROOT / "datasets" / DS_NAME if ROOT else None
if DRIVE_DS and DRIVE_DS.exists():
    print(f"from Drive: {drive.pull(DRIVE_DS, LOCAL)} files copied")
srcs = [s.strip() for s in SRC.split(",") if s.strip()]
if UPLOAD_ZIP:
    from google.colab import files
    up = paths.WORK / "uploads"
    up.mkdir(parents=True, exist_ok=True)
    for name, data in files.upload().items():  # a file picker opens below the cell
        (up / name).write_bytes(data)
        srcs.append(str(up / name))
if srcs:
    nb.run(["-m", "sdxlr.data", "--dest", LOCAL, *srcs, "--res", 512, "--kind", "zimage"])
    for s in srcs:  # uploads are copied into the dataset; free the disk
        if s.startswith(str(paths.WORK / "uploads")):
            Path(s).unlink(missing_ok=True)
imgs = sorted(p for p in LOCAL.iterdir() if p.suffix.lower() in IMGS)
assert imgs, "no images: tick UPLOAD_ZIP, fill SRC, or check DS_NAME"
missing = [p for p in imgs if not p.with_suffix(".txt").exists()]
print(f"{len(imgs)} images, {len(imgs) - len(missing)} captioned")
if missing and CAPTION_MISSING:
    args = ["-m", "sdxlr.caption", "--images", LOCAL, "--style", "id", "--target", "zimage",
            "--trigger", TRIGGER, "--class-word", CLASS_WORD]
    nb.run(args)
if DRIVE_DS:
    print(f"to Drive: {drive.pull(LOCAL, DRIVE_DS)} files copied")

# 2. Settings
Defaults are the lightest setup: **fast** (Turbo + ostris training adapter, 512 px, fp8 weights, 6 epochs) with the
**abliterated** text encoder; tests run on Turbo 4-bit (about 6 GB VRAM). `best` = Z-Image Base, 10 epochs (measured).
Any photo count works: repeats and epochs scale so the total stays near the proven amount. For 100 photos that is
1 repeat x 6 epochs = 600 steps on fast (about 62 min est), or 1 x 4 = 400 steps on best (about 43 min).
**TE**: the text encoder used for training. Generate with the same one (the Z-Image app has the same list).
Leave EPOCHS / REPEATS / DIM / LR at 0 for the preset values; repeats 0 = auto from your photo count.

In [ ]:
#@title Settings
SUBJECT = "person"  #@param ["person", "character/object", "style", "concept/pose"]
PRESET = "fast"  #@param ["fast", "best"]
TE = "abliterated"  #@param ["abliterated", "official", "heretic", "engineer"]
RUN = "my_zlora"  #@param {type:"string"}
RESOLUTION = 512  #@param [512, 768]
EPOCHS = 0  #@param {type:"integer"}
REPEATS = 0  #@param {type:"integer"}
DIM = 0  #@param {type:"integer"}
LR = 0.0  #@param {type:"number"}
RESUME = True  #@param {type:"boolean"}
#@markdown RESUME continues a stopped run with the same RUN name from its state on Drive. 768 px is about 2.3x slower (est).
import yaml
cfg = yaml.safe_load((REPO / "configs/train" / ("zimage_person.yaml" if PRESET == "best" else "zimage_fast.yaml")).read_text())
sets = [f"repeats={REPEATS}", f"resolution={RESOLUTION}"]
if EPOCHS: sets.append(f"epochs={EPOCHS}")
if DIM: sets += [f"network_dim={DIM}", f"network_alpha={DIM}"]  # alpha = dim, as the preset
if LR: sets.append(f"lr={LR}")
n = len(sorted(p for p in LOCAL.iterdir() if p.suffix.lower() in IMGS))
p = plan.train("zimage", SUBJECT, n, RESOLUTION, cfg["batch_size"], EPOCHS or cfg["epochs"], REPEATS)
print(p.summary())

# 3. Train
Caches latents and text-encoder outputs first, then trains. Every epoch goes to `SDXLR/loras/zimage/<RUN>/`
(the `_diffusers` files are what the app loads) and the newest state to `SDXLR/state/<RUN>/`.
If Colab disconnects, run Setup, Dataset, Settings and this cell again: it continues where it stopped.

In [ ]:
#@title Train
args = ["-m", "sdxlr.train_zimage", "--name", RUN, "--images", LOCAL, "--preset", PRESET, "--te", TE,
        "--subject", SUBJECT, "--trigger", TRIGGER, "--class-word", CLASS_WORD, "--set", *sets]
if ROOT:
    args += ["--drive", ROOT] + (["--resume"] if RESUME else [])
nb.run(args)

# 4. Test
A grid: rows = the last epochs (x prompts), columns = LoRA strengths, same seed everywhere.
Runs on Z-Image Turbo 4-bit with the training TE: about 36 s per image plus the model load.

In [ ]:
#@title Test
EPOCHS_TO_TEST = "last:3"  #@param {type:"string"}
#@markdown last:K, all, or a list like 4,7,10
STRENGTHS = "0.6 0.8 1.0"  #@param {type:"string"}
PROMPT_1 = "A photo of {t} {c} sitting by a window, soft daylight on the face, relaxed expression, shallow depth of field"  #@param {type:"string"}
PROMPT_2 = "A full-body photo of {t} {c} walking down a busy city street at dusk, neon signs, candid"  #@param {type:"string"}
STEPS = 9  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}
prompts = [x.replace("{t}", TRIGGER).replace("{c}", CLASS_WORD) for x in (PROMPT_1, PROMPT_2) if x.strip()]
run_dir = paths.WORK / "runs" / RUN
if ROOT and not any(run_dir.glob("*.safetensors")):  # a new VM: test what Drive has
    drive.pull(ROOT / "loras" / "zimage" / RUN, run_dir)
args = ["-m", "sdxlr.compare", "--name", RUN, "--zimage", "--model", "turbo 4-bit (unsloth)", "--te", TE, "--cfg", 0,
        "--lora-dir", run_dir, "--epochs", EPOCHS_TO_TEST, "--strengths", *STRENGTHS.split(),
        "--prompts", *prompts, "--steps", STEPS, "--seed", SEED]
if ROOT:
    args += ["--drive-out", ROOT / "compare" / RUN]
nb.run(args)
from IPython.display import Image as _Img, display
display(_Img(filename=str(paths.WORK / "compare" / RUN / "grid.png")))

# 5. Pick
Copies the epoch you like to `SDXLR/loras/zimage/<RUN>_e<N>.safetensors` (with its trigger-word sidecar).
The Z-Image app lists that folder.

In [ ]:
#@title Pick an epoch
PICK_EPOCH = 0  #@param {type:"integer"}
from sdxlr import loraformat
dst = loraformat.pick(run_dir, PICK_EPOCH, (ROOT or paths.WORK) / "loras" / "zimage", RUN)
print(f"saved {dst}")

# Finish
Drive writes back lazily: run this before closing the tab, then **Runtime > Disconnect and delete runtime**.

In [ ]:
#@title Save to Drive and finish
if ROOT:
    drive.flush_and_unmount()